#### ENRICH `ORDER_REVIEWS` FROM BRONZE TO SILVER

In [0]:
import uuid, traceback
from datetime import datetime
from pyspark.sql import functions as F

In [0]:
CATALOG = 'ORDERS_SALES'
SCHEMA = 'SILVER'
TARGET_TABLE = 'ORDERS_REVIEWS_ENRICHED'
TARGET = f"{CATALOG}.{SCHEMA}.{TARGET_TABLE}"
SOURCE = f"{CATALOG}.BRONZE.ORDERS_REVIEWS"
AUDIT_TABLE = f"{CATALOG}.AUDIT.PIPELINE_AUDIT_LOG"

In [0]:
order_reviews_df = spark.table(SOURCE)
order_reviews_df.limit(1).display()

In [0]:
from pyspark.sql.functions import *

order_reviews_trans_df = (
    order_reviews_df.select(
        "review_id",
        "order_id",
        "review_score",
        "review_comment_title",
        "review_comment_message",
        col("file_path").alias("origin_file_path"),
        current_timestamp().alias("load_timestamp"),
    ).where("review_comment_title IS NOT NULL OR review_comment_message IS NOT NULL")
)
order_reviews_trans_df.limit(1).display()

In [0]:
order_reviews_trans_df.createOrReplaceTempView('order_reviews_trans_temp_vw')

#### INCREMENTAL LOAD `ORDERS_SALES.SILVER.ORDERS_REVIEWS_ENRICHED`

In [0]:
from pyspark.sql.functions import *

def write_audit(run_id, status, start_time, rows=None, error=None):
  end = datetime.now()
  row = (run_id, "ORDERS_REVIEWS_ENRICHED_LOAD", "SILVER", SOURCE, TARGET, status, rows,
           start_time, end, (end - start_time).total_seconds(),
           error[:4000] if error else None,
           spark.sql("SELECT current_user()").first()[0])
  (spark.createDataFrame([row], spark.table(AUDIT_TABLE).schema)
          .write.mode("append").format("delta").saveAsTable(AUDIT_TABLE))

run_id, start_time = str(uuid.uuid4()), datetime.now()
status, rows_written, err = "SUCCESS", None, None


try:
  result = spark.sql(f'''MERGE INTO {CATALOG}.{SCHEMA}.{TARGET_TABLE} tgt
USING (
  SELECT *
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY review_id
               ORDER BY load_timestamp DESC
           ) rn
    FROM order_reviews_trans_temp_vw
  )
  WHERE rn = 1
) src
ON (src.review_id = tgt.review_id)
WHEN MATCHED
    THEN UPDATE SET
    tgt.order_id = src.order_id,
    tgt.review_score = src.review_score,
    tgt.review_comment_title = src.review_comment_title,
    tgt.review_comment_message = src.review_comment_message
WHEN NOT MATCHED THEN
INSERT (review_id, order_id, review_score, review_comment_title, review_comment_message, origin_file_path,
load_timestamp)
VALUES (src.review_id, src.order_id, src.review_score, src.review_comment_title, src.review_comment_message, src.origin_file_path, current_timestamp)
''')
  
  m = result.first().asDict()
  inserted = m.get("num_inserted_rows") or 0
  updated  = m.get("num_updated_rows") or 0
  rows_written = int(inserted + updated)
  print(f"inserted={inserted}, updated={updated}")

except Exception:
  status, err = "FAILED", traceback.format_exc()
  raise
finally:
  write_audit(run_id, status, start_time, rows_written, err)

In [0]:
dbutils.notebook.exit('SUCCESSFULLY LOADED INTO ORDERS_SALES.SILVER.ORDERS_REVIEWS_ENRICHED')